#### Keels locations 

In [25]:
import pandas as pd 
import numpy as np 
from dotenv import load_dotenv
import os
import requests

In [4]:
_ = load_dotenv()

In [6]:
google_api = os.getenv('GOOGLE_MAPS_API_KEY')

In [38]:
def get_gcp_gps(address):
    """
    Geocodes an address and returns (latitude, longitude, district) as a tuple.
    
    Args:
        address (str): The address to geocode.
    
    Returns:
        pd.Series: Series with latitude, longitude, and district.
    """
    try:
        address += ", Sri lanka"
        # Construct the URL
        url = f"https://maps.googleapis.com/maps/api/geocode/json?address={address}&key={google_api}"

        # Send the request
        response = requests.get(url)

        # Parse the JSON response
        if response.status_code == 200:
            data = response.json()
            if data['status'] == 'OK' and data['results']:
                location = data['results'][0]['geometry']['location']
                latitude = location['lat']
                longitude = location['lng']
                district = None
                for component in data['results'][0]['address_components']:
                    if 'administrative_area_level_2' in component['types']:
                        district = component['long_name']
                        break
                # Fallback to administrative_area_level_1 if level_2 is not found
                if not district:
                    for component in data['results'][0]['address_components']:
                        if 'administrative_area_level_1' in component['types']:
                            district = component['long_name']
                            break
                # If still no district, use a default value
                district = district if district else 'Unknown'
                print(f"Address: {address}, Latitude: {latitude}, Longitude: {longitude}, District: {district}")
                return pd.Series([latitude, longitude, district])
            else:
                print(f"Geocoding error for address '{address}': {data.get('status', 'Unknown')}")
                return pd.Series([None, None, 'Unknown'])
        else:
            print(f"HTTP Error for address '{address}': {response.status_code}")
            return pd.Series([None, None, 'Unknown'])
    except Exception as e:
        print(f"Exception for address '{address}': {e}")
        return pd.Series([None, None, 'Unknown'])

In [31]:
keels_master = pd.read_csv('../data/test/keels_master.csv', encoding='latin1')

In [32]:
keels_master.head()

,CODE,LOCATION,ADDRESS
0,S2HK,Kottawa 2,"No. 191, Horan Road, Kottawa."
1,SCA2,Athurugiriya 2,"No. 175/12, Malambe Road, Athurugiriya"
2,SCAR,Arangala,"No. 730, Malabe Road, Arangala, Athurugiriya"
3,SCBL,Boralasgomuwa,"No. 105/1, Kesbewa Road, Katuwala, Boralasgomuwa"
4,SCCC,Crescat,"No. 89, Crescat Boulevard, Galle Road, Colombo 03"


In [33]:
keels_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135 entries, 0 to 134
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   CODE      135 non-null    object
 1   LOCATION  135 non-null    object
 2   ADDRESS   135 non-null    object
dtypes: object(3)
memory usage: 3.3+ KB


In [34]:
get_gcp_gps(keels_master['ADDRESS'][0])

Exception for address 'No. 191, Horan Road, Kottawa., Sri lanka': name 'gcp_key' is not defined


0       None
1       None
2    Unknown
dtype: object

In [39]:
keels_master[['LATITUDE','LONGITUDE', 'DISTRICT']] = keels_master['ADDRESS'].apply(get_gcp_gps)

Address: No. 191, Horan Road, Kottawa., Sri lanka, Latitude: 6.8411652, Longitude: 79.9654324, District: Colombo
Address: No. 175/12, Malambe Road, Athurugiriya, Sri lanka, Latitude: 6.8723185, Longitude: 80.0003875, District: Colombo
Address: No. 730, Malabe Road, Arangala, Athurugiriya, Sri lanka, Latitude: 6.851618999999999, Longitude: 79.9684822, District: Colombo
Address: No. 105/1, Kesbewa Road, Katuwala, Boralasgomuwa, Sri lanka, Latitude: 6.834556099999999, Longitude: 79.9047633, District: Colombo
Address: No. 89, Crescat Boulevard, Galle Road, Colombo 03, Sri lanka, Latitude: 6.917104399999999, Longitude: 79.84818709999999, District: Colombo
Address: No. 531, Capital Mall, Madiwela Road, Thalawathugoda, Sri lanka, Latitude: 6.8733711, Longitude: 79.9171546, District: Colombo
Address: No. 268, Sir D.B. Jayathilake Mawatha, (Hill Street), Dehiwala, Sri lanka, Latitude: 6.8501667, Longitude: 79.8725682, District: Colombo
Address: No. 303, Old Kottawa Road, Embuldeniya, Sri lanka,

In [52]:
nan_values = keels_master[['LATITUDE', 'LONGITUDE', 'DISTRICT']].isna().any(axis=1)

In [55]:
# Apply get_gcp_gps to the LOCATION column for those rows
keels_master.loc[nan_values, ['LATITUDE', 'LONGITUDE', 'DISTRICT']] = (
    keels_master.loc[nan_values, 'LOCATION'].apply(get_gcp_gps)
)

Address: Wijerama, Sri lanka, Latitude: 6.855181699999999, Longitude: 79.9087754, District: Colombo
Address: Aluthgama, Sri lanka, Latitude: 6.4345732, Longitude: 80.0003875, District: Kalutara
Address: Puttalam, Sri lanka, Latitude: 8.0407913, Longitude: 79.839386, District: Puttalam


In [58]:
keels_master.drop_duplicates(inplace=True)

In [76]:
keels_master['BRAND'] = "Keels"

In [77]:
keels_master.to_csv('../data/test/keels_master.csv', index=False)

In [63]:
po_file = pd.read_csv('../data/test/orders-vol/14-07-2025-PO-2.csv')

In [88]:
merged = po_file.merge(
    keels_master[['CODE', 'LOCATION', 'ADDRESS', 'LATITUDE', 'LONGITUDE', 'BRAND', 'DISTRICT']],
    on='CODE',
    how='left',  # Keep all rows from po_file
    suffixes=('', '_master')
)

In [91]:
po_file['LOCATION'] = po_file['LOCATION'].combine_first(merged['LOCATION_master'])
po_file['ADDRESS'] = po_file['ADDRESS'].combine_first(merged['ADDRESS_master'])
po_file['LATITUDE'] = po_file['LATITUDE'].combine_first(merged['LATITUDE_master'])
po_file['LONGITUDE'] = po_file['LONGITUDE'].combine_first(merged['LONGITUDE_master'])
po_file['BRAND'] = po_file['BRAND'].combine_first(merged['BRAND_master'])
po_file['DISTRICT'] = po_file['DISTRICT'].combine_first(merged['DISTRICT_master'])

# Verify the result (check remaining NaN values)
print(po_file[['LOCATION', 'ADDRESS', 'LATITUDE', 'LONGITUDE', 'BRAND', 'DISTRICT']].isna().sum())

LOCATION     15
ADDRESS      15
LATITUDE     15
LONGITUDE    15
BRAND        15
DISTRICT     15
dtype: int64


In [93]:
po_file.to_csv('../data/test/orders-vol/14-07-2025-PO-4.csv', index=False)